# 00 — Kiểm tra môi trường Colab
Chạy tuần tự trên runtime GPU từ phiên sạch. Không cài lại PyTorch, không tải mô hình hoặc kiểm thử SFT/DPO.

**Nguồn thông tin:** người dùng đã đo trong một phiên Colab trước: Tesla T4, 15.360 MiB = 15 GiB, torch `2.14.1+cu130`, CUDA build `13.0`, CUDA khả dụng, nhân ma trận thành công. Notebook không coi các giá trị này là kết quả hiện tại hay yêu cầu phiên bản phải trùng.

**Đường dẫn:** notebook độc lập với mã repo; có thể mở trực tiếp từ repo trên Colab. `REPO_DIR` là tùy chọn để ghi code revision nếu đã clone/copy repo. Chưa có đường dẫn Drive được chốt. Mặc định `/content` mất khi ngắt phiên: tải báo cáo xuống hoặc bật Drive và đặt `OUTPUT_DIR` bên dưới `/content/drive/MyDrive/`. Không ghi token vào ô hoặc output.


In [ ]:
from pathlib import Path
import os

MOUNT_DRIVE = False
REPO_DIR = None  # Đặt đường dẫn repo đã có nếu muốn ghi commit; không tự clone.
OUTPUT_DIR = Path('/content/environment_checks') if Path('/content').exists() else Path('environment_checks')
SEED = 42

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
# Nếu bật Drive, sửa OUTPUT_DIR ở trên thành đường dẫn lưu bền vững của bạn.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Report directory:', OUTPUT_DIR.resolve())
print('Persistence: verify location or download files before disconnecting.')


In [ ]:
import datetime
import importlib.metadata
import importlib.util
import platform
import random
import subprocess
import sys
import uuid
import json

def command_result(args, cwd=None):
    try:
        result = subprocess.run(args, cwd=cwd, capture_output=True, text=True, timeout=60)
        return {'returncode': result.returncode, 'stdout': result.stdout.strip(), 'stderr': result.stderr.strip()}
    except (OSError, subprocess.TimeoutExpired) as exc:
        return {'error': str(exc)}

report = {
    'observed_at_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'evidence_origin': 'executed_in_current_runtime',
    'python': sys.version,
    'platform': platform.platform(),
    'seed': SEED,
    'user_reported_prior_colab': {
        'evidence_origin': 'user_report_only_not_verified_here',
        'gpu': 'Tesla T4', 'vram_mib': 15360,
        'torch': '2.14.1+cu130', 'torch_cuda_build': '13.0',
        'cuda_available': True, 'cuda_matmul_passed': True,
    },
    'nvidia_smi': command_result(['nvidia-smi']),
    'packages': {},
}
for package in ['torch', 'transformers', 'peft', 'trl', 'accelerate', 'bitsandbytes', 'datasets']:
    try:
        report['packages'][package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        report['packages'][package] = None
if REPO_DIR is not None:
    report['code_revision'] = command_result(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR)
    report['code_worktree'] = command_result(['git', 'status', '--porcelain'], cwd=REPO_DIR)
print(json.dumps(report, ensure_ascii=False, indent=2))


In [ ]:
random.seed(SEED)
report['gpu_smoke'] = {'status': 'not_run', 'reason': 'PyTorch is not installed'}
if importlib.util.find_spec('torch') is not None:
    try:
        import torch
        torch.manual_seed(SEED)
        report['torch_runtime'] = {
            'version': torch.__version__,
            'cuda_build': torch.version.cuda,
            'cuda_available': torch.cuda.is_available(),
        }
        if not torch.cuda.is_available():
            report['gpu_smoke'] = {'status': 'not_run', 'reason': 'CUDA unavailable; select GPU runtime in Colab'}
        else:
            torch.cuda.manual_seed_all(SEED)
            device = torch.device('cuda:0')
            properties = torch.cuda.get_device_properties(device)
            torch.cuda.reset_peak_memory_stats(device)
            # Integer-valued float32 inputs give an exact small reference product.
            a_cpu = torch.tensor([[1., 2.], [3., 4.]], dtype=torch.float32)
            b_cpu = torch.tensor([[5., 6.], [7., 8.]], dtype=torch.float32)
            expected = torch.tensor([[19., 22.], [43., 50.]], dtype=torch.float32)
            a, b = a_cpu.to(device), b_cpu.to(device)
            result = a @ b
            torch.cuda.synchronize(device)
            torch.testing.assert_close(result.cpu(), expected, rtol=0, atol=0)
            report['gpu_smoke'] = {
                'status': 'passed', 'device': str(device), 'gpu_name': properties.name,
                'total_memory_mib': properties.total_memory / 2**20,
                'compute_capability': list(torch.cuda.get_device_capability(device)),
                'peak_allocated_mib': torch.cuda.max_memory_allocated(device) / 2**20,
                'matmul_result': result.cpu().tolist(),
            }
            del a, b, result
            torch.cuda.empty_cache()
    except Exception as exc:
        report['gpu_smoke'] = {'status': 'failed', 'error': repr(exc)}
print(json.dumps(report['gpu_smoke'], ensure_ascii=False, indent=2))


In [ ]:
run_id = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8]
report_path = OUTPUT_DIR / f'environment-{run_id}.json'
freeze_path = OUTPUT_DIR / f'packages-{run_id}.txt'
# pip freeze records installed packages; it does not install or upgrade anything.
freeze = command_result([sys.executable, '-m', 'pip', 'freeze'])
report['pip_freeze'] = {'returncode': freeze.get('returncode'), 'error': freeze.get('error'), 'stderr': freeze.get('stderr')}
freeze_path.write_text(freeze.get('stdout', '') + '\n', encoding='utf-8')
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print('Saved:', report_path.resolve())
print('Saved:', freeze_path.resolve())
if report['gpu_smoke']['status'] == 'failed':
    raise RuntimeError('GPU smoke failed; inspect the saved report before changing packages.')
if report['gpu_smoke']['status'] == 'not_run':
    print('GPU NOT VERIFIED. CPU-only execution does not meet the Colab GPU check.')


## Diễn giải và bước tiếp theo
- `passed` chỉ xác nhận phép nhân ma trận nhỏ bằng CUDA trong phiên hiện tại, không xác nhận tương thích PEFT/TRL/bitsandbytes hay đủ VRAM huấn luyện.
- `not_run` cần chạy trên Colab GPU; `failed` cần xem báo cáo, không tự nâng cấp PyTorch để che lỗi.
- Giữ báo cáo môi trường cùng manifest thí nghiệm; kiểm tra nội dung đường dẫn riêng trước khi chia sẻ. Không commit toàn bộ `pip freeze` không cần thiết.
- Pilot SFT và DPO riêng: đo peak VRAM, thời gian, chiều dài token, batch, tích lũy gradient, dtype và cách giữ reference SFT đóng băng. Chỉ chốt phiên bản thư viện sau khi đọc tài liệu chính thức và thử thực tế.
- Checkpoint huấn luyện sau này phải lưu bền vững adapter, optimizer, scheduler, RNG, bước hiện tại, cấu hình và định danh dữ liệu/code; thử tiếp tục sau ngắt. Notebook này chỉ lưu báo cáo môi trường, chưa triển khai huấn luyện.
